# SQL practice 12: query plans and performance

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-12-query-plans-and-performance.ipynb)

12 exercises that pair a correct query with a checked property of its plan: pruning, pushdown, join strategy and shuffles. Each exercise names the [reference entries](sql-reference-12-query-plans-and-performance.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 12](sql-reference-12-query-plans-and-performance.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. This notebook also writes Parquet copies of the data: `orders_pq` (partitioned by `order_date`), `events_pq` and `items_pq`, and defines `plan_lines(query, *keywords)`, which filters the `EXPLAIN FORMATTED` output.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

# File-backed copies of the sample data, so plans show scans, pruning and pushdown.
for table, ddl in {
    "orders_pq": "USING parquet PARTITIONED BY (order_date) AS SELECT order_id, customer_id, status, order_ts, amount, to_date(order_ts) AS order_date FROM orders",
    "events_pq": "USING parquet AS SELECT * FROM events",
    "items_pq": "USING parquet AS SELECT * FROM order_items",
}.items():
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    spark.sql(f"CREATE TABLE {table} {ddl}")


def plan_lines(query, *keywords):
    """The lines of EXPLAIN FORMATTED that contain any of the keywords."""
    return [line.strip() for line in plan(query).splitlines() if any(k in line for k in keywords)]

## Exercises

7 easy, 4 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Read only the columns you need](#exercise-1-read-only-the-columns-you-need) | Easy | [Partition pruning, filter pushdown and column pruning](sql-reference-12-query-plans-and-performance.ipynb#partition-pruning-filter-pushdown-and-column-pruning) |
| 2 | [Prune partitions](#exercise-2-prune-partitions) | Easy | [Partition pruning, filter pushdown and column pruning](sql-reference-12-query-plans-and-performance.ipynb#partition-pruning-filter-pushdown-and-column-pruning) |
| 3 | [A sargable time filter](#exercise-3-a-sargable-time-filter) | Medium | [Partition pruning, filter pushdown and column pruning](sql-reference-12-query-plans-and-performance.ipynb#partition-pruning-filter-pushdown-and-column-pruning) |
| 4 | [Semi join instead of join plus DISTINCT](#exercise-4-semi-join-instead-of-join-plus-distinct) | Easy | [Join strategies](sql-reference-12-query-plans-and-performance.ipynb#join-strategies) |
| 5 | [Force a broadcast join](#exercise-5-force-a-broadcast-join) | Medium | [Join strategies](sql-reference-12-query-plans-and-performance.ipynb#join-strategies) |
| 6 | [Top-N without a full sort](#exercise-6-top-n-without-a-full-sort) | Easy | [Physical operators](sql-reference-12-query-plans-and-performance.ipynb#physical-operators) |
| 7 | [UNION ALL when duplicates cannot occur](#exercise-7-union-all-when-duplicates-cannot-occur) | Easy | [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations) |
| 8 | [Approximate distinct users](#exercise-8-approximate-distinct-users) | Easy | [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations) |
| 9 | [Running totals with a window, not a self join](#exercise-9-running-totals-with-a-window-not-a-self-join) | Medium | [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations) |
| 10 | [Aggregate before joining](#exercise-10-aggregate-before-joining) | Medium | [Performance tuning checklist](sql-reference-12-query-plans-and-performance.ipynb#performance-tuning-checklist) |
| 11 | [Partition the window](#exercise-11-partition-the-window) | Easy | [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations) |
| 12 | [Salting a skewed aggregation](#exercise-12-salting-a-skewed-aggregation) | Hard | [Join strategies](sql-reference-12-query-plans-and-performance.ipynb#join-strategies) |

## Exercise 1: Read only the columns you need

**Difficulty:** Easy · about 5 min · **Practises:** [Partition pruning, filter pushdown and column pruning](sql-reference-12-query-plans-and-performance.ipynb#partition-pruning-filter-pushdown-and-column-pruning)

*Columnar formats read nothing you do not select.*

From `orders_pq`, return `order_id` and `amount` of orders above 200, ordered by `order_id`. The plan must read only those two columns and push the amount filter into the scan.

**Expected output**

| order_id | amount |
| --- | --- |
| 4 | 260.00 |
| 7 | 310.00 |
| 9 | 1180.00 |
| 11 | 210.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, amount
    FROM orders_pq
    WHERE amount > 200
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Prune partitions

**Difficulty:** Easy · about 5 min · **Practises:** [Partition pruning, filter pushdown and column pruning](sql-reference-12-query-plans-and-performance.ipynb#partition-pruning-filter-pushdown-and-column-pruning)

*Filtering on the partition column skips whole directories.*

From `orders_pq`, return `order_date` and `orders` for 2026-01-12 onwards, ordered by `order_date`, filtering on the partition column.

**Expected output**

| order_date | orders |
| --- | --- |
| 2026-01-12 | 2 |
| 2026-01-13 | 1 |
| 2026-01-14 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_date, count(*) AS orders
    FROM orders_pq
    WHERE order_date >= DATE'2026-01-12'
    GROUP BY order_date
    ORDER BY order_date
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: A sargable time filter

**Difficulty:** Medium · about 10 min · **Practises:** [Partition pruning, filter pushdown and column pruning](sql-reference-12-query-plans-and-performance.ipynb#partition-pruning-filter-pushdown-and-column-pruning)

*Functions on a data column stop the scan from skipping row groups.*

From `events_pq`, return `user_id`, `event` and `ts` for events on 2026-01-12, ordered by `ts`, `event`. Write the filter so it is pushed into the scan (no function around `ts`).

**Expected output**

| user_id | event | ts |
| --- | --- | --- |
| 5 | view | 2026-01-12 09:00:00 |
| 5 | view | 2026-01-12 09:00:00 |
| 5 | purchase | 2026-01-12 12:30:00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT user_id, event, ts
    FROM events_pq
    WHERE ts >= TIMESTAMP'2026-01-12 00:00:00' AND ts < TIMESTAMP'2026-01-13 00:00:00'
    ORDER BY ts, event
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Semi join instead of join plus DISTINCT

**Difficulty:** Easy · about 5 min · **Practises:** [Join strategies](sql-reference-12-query-plans-and-performance.ipynb#join-strategies)

*EXISTS is planned as a semi join: no duplicates and no extra shuffle.*

Return `customer_id` and `name` of customers with at least one order in `orders_pq`, ordered by `customer_id`, so that the plan uses a semi join.

**Expected output**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Bright Retail |
| 3 | Cyan Traders |
| 4 | Delta Grocers |
| 5 | Echo Supplies |
| 6 | Foxtrot Foods |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    WHERE EXISTS (SELECT 1 FROM orders_pq o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Force a broadcast join

**Difficulty:** Medium · about 10 min · **Practises:** [Join strategies](sql-reference-12-query-plans-and-performance.ipynb#join-strategies)

*Hints override the optimiser when estimates are wrong.*

Automatic broadcasting is switched off in the input cell. Return `order_id`, `name` and `amount` for orders in `orders_pq` over 200 joined to `customers`, ordered by `order_id`, with a hint so the plan broadcasts `customers`.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # every table now counts as large

**Expected output**

| order_id | name | amount |
| --- | --- | --- |
| 4 | Bright Retail | 260.00 |
| 7 | Cyan Traders | 310.00 |
| 9 | Delta Grocers | 1180.00 |
| 11 | Foxtrot Foods | 210.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT /*+ BROADCAST(c) */ o.order_id, c.name, o.amount
    FROM orders_pq o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.amount > 200
    ORDER BY o.order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Top-N without a full sort

**Difficulty:** Easy · about 3 min · **Practises:** [Physical operators](sql-reference-12-query-plans-and-performance.ipynb#physical-operators)

*ORDER BY with LIMIT becomes a per-task top-n.*

Return `order_id` and `amount` of the three largest orders in `orders_pq` (ties by `order_id`).

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")   # back to the default

**Expected output**

| order_id | amount |
| --- | --- |
| 9 | 1180.00 |
| 7 | 310.00 |
| 4 | 260.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, amount
    FROM orders_pq
    ORDER BY amount DESC, order_id
    LIMIT 3
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: UNION ALL when duplicates cannot occur

**Difficulty:** Easy · about 3 min · **Practises:** [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations)

*UNION de-duplicates with a shuffle you may not need.*

Return `kind` and `id` listing paid orders (`kind = 'order'`, the order id) and customers without a region (`kind = 'customer'`, the customer id) in one result, ordered by `kind`, `id`, without a de-duplication step.

**Expected output**

| kind | id |
| --- | --- |
| customer | 4 |
| order | 1 |
| order | 4 |
| order | 7 |
| order | 10 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT 'customer' AS kind, customer_id AS id FROM customers WHERE region IS NULL
    UNION ALL
    SELECT 'order' AS kind, order_id AS id FROM orders_pq WHERE status = 'paid'
    ORDER BY kind, id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Approximate distinct users

**Difficulty:** Easy · about 5 min · **Practises:** [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations)

*HyperLogLog sketches avoid the cost of exact distinct counts at scale.*

From `events_pq`, return `day` and `approx_users` (approximate distinct users), ordered by `day`.

**Expected output**

| day | approx_users |
| --- | --- |
| 2026-01-06 | 2 |
| 2026-01-07 | 1 |
| 2026-01-08 | 1 |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-11 | 1 |
| 2026-01-12 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT to_date(ts) AS day, approx_count_distinct(user_id) AS approx_users
    FROM events_pq
    GROUP BY to_date(ts)
    ORDER BY day
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Running totals with a window, not a self join

**Difficulty:** Medium · about 10 min · **Practises:** [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations)

*A self-join running total is quadratic; a window is one pass per partition.*

From `orders_pq`, return `customer_id`, `order_id` and `running_total` (cumulative amount per customer by `order_ts`, row by row) for customers 1 to 3, ordered by `customer_id`, `order_id`, using a window function.

**Expected output**

| customer_id | order_id | running_total |
| --- | --- | --- |
| 1 | 1 | 120.00 |
| 1 | 2 | 204.50 |
| 1 | 3 | 234.50 |
| 2 | 4 | 260.00 |
| 2 | 5 | 315.25 |
| 3 | 6 | 149.00 |
| 3 | 7 | 459.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, order_id,
           sum(amount) OVER (PARTITION BY customer_id ORDER BY order_ts
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
    FROM orders_pq
    WHERE customer_id <= 3
    ORDER BY customer_id, order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Aggregate before joining

**Difficulty:** Medium · about 10 min · **Practises:** [Performance tuning checklist](sql-reference-12-query-plans-and-performance.ipynb#performance-tuning-checklist)

*Joining aggregated rows moves far less data than joining raw rows.*

Return `region` (`Unknown` when missing), `orders` and `revenue` (from `orders_pq`, excluding returns), by aggregating orders per customer **before** joining `customers`. Order by `revenue` descending.

**Expected output**

| region | orders | revenue |
| --- | --- | --- |
| Unknown | 1 | 1180.00 |
| North | 4 | 519.75 |
| South | 2 | 459.00 |
| West | 3 | 380.40 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH per_customer AS (
        SELECT customer_id, count(*) AS orders, sum(amount) AS revenue
        FROM orders_pq
        WHERE status <> 'returned'
        GROUP BY customer_id
    )
    SELECT coalesce(c.region, 'Unknown') AS region, sum(p.orders) AS orders, sum(p.revenue) AS revenue
    FROM per_customer p
    JOIN customers c ON c.customer_id = p.customer_id
    GROUP BY coalesce(c.region, 'Unknown')
    ORDER BY revenue DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Partition the window

**Difficulty:** Easy · about 5 min · **Practises:** [Aggregations, distinct counts and set operations](sql-reference-12-query-plans-and-performance.ipynb#aggregations-distinct-counts-and-set-operations)

*A window without PARTITION BY sends every row to one task.*

From `events_pq`, return `user_id`, `ts` and `event_no` (the event's position in the user's own timeline, counting every row) for users 4 and 5, ordered by `user_id`, `event_no`. The plan must not gather all rows into a single partition.

**Expected output**

| user_id | ts | event_no |
| --- | --- | --- |
| 4 | 2026-01-09 20:05:00 | 1 |
| 4 | 2026-01-10 08:30:00 | 2 |
| 5 | 2026-01-11 14:45:00 | 1 |
| 5 | 2026-01-12 09:00:00 | 2 |
| 5 | 2026-01-12 09:00:00 | 3 |
| 5 | 2026-01-12 12:30:00 | 4 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT user_id, ts, row_number() OVER (PARTITION BY user_id ORDER BY ts, event) AS event_no
    FROM events_pq
    WHERE user_id IN (4, 5)
    ORDER BY user_id, event_no
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Salting a skewed aggregation

**Difficulty:** Hard · about 20 min · **Practises:** [Join strategies](sql-reference-12-query-plans-and-performance.ipynb#join-strategies)

*When one key holds most rows, two-phase salted aggregation spreads the work.*

Using `clicks` below (customer 1 is extremely hot), compute clicks per `customer_id` in two phases: first group by customer and a salt (`pmod(click_id, 8)`), then sum the partial counts per customer. Return `customer_id` and `clicks`, ordered by `customer_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW clicks AS
SELECT id AS click_id, CASE WHEN id % 10 < 8 THEN 1 ELSE CAST(id % 4 AS INT) + 2 END AS customer_id
FROM range(10000)
""")
show("SELECT customer_id, count(*) AS clicks FROM clicks GROUP BY customer_id ORDER BY customer_id")

**Expected output**

| customer_id | clicks |
| --- | --- |
| 1 | 8000 |
| 2 | 500 |
| 3 | 500 |
| 4 | 500 |
| 5 | 500 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH salted AS (
        SELECT customer_id, pmod(click_id, 8) AS salt, count(*) AS partial_clicks
        FROM clicks
        GROUP BY customer_id, pmod(click_id, 8)
    )
    SELECT customer_id, sum(partial_clicks) AS clicks
    FROM salted
    GROUP BY customer_id
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 12: query plans and performance](sql-reference-12-query-plans-and-performance.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).